In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/spaceship-titanic/sample_submission.csv
/kaggle/input/competitions/spaceship-titanic/train.csv
/kaggle/input/competitions/spaceship-titanic/test.csv


In [2]:
import pandas as pd
import numpy as np

train = pd.read_csv("/kaggle/input/competitions/spaceship-titanic/train.csv")
test = pd.read_csv("/kaggle/input/competitions/spaceship-titanic/test.csv")

In [3]:
train.shape

(8693, 14)

In [4]:
test.shape

(4277, 13)

In [5]:
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8693 entries, 0 to 8692
Data columns (total 14 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   PassengerId   8693 non-null   object 
 1   HomePlanet    8492 non-null   object 
 2   CryoSleep     8476 non-null   object 
 3   Cabin         8494 non-null   object 
 4   Destination   8511 non-null   object 
 5   Age           8514 non-null   float64
 6   VIP           8490 non-null   object 
 7   RoomService   8512 non-null   float64
 8   FoodCourt     8510 non-null   float64
 9   ShoppingMall  8485 non-null   float64
 10  Spa           8510 non-null   float64
 11  VRDeck        8505 non-null   float64
 12  Name          8493 non-null   object 
 13  Transported   8693 non-null   bool   
dtypes: bool(1), float64(6), object(7)
memory usage: 891.5+ KB


In [6]:
print(train["Transported"].value_counts())
print(train["Transported"].value_counts(normalize=True))

Transported
True     4378
False    4315
Name: count, dtype: int64
Transported
True     0.503624
False    0.496376
Name: proportion, dtype: float64


In [7]:
train.isna().mean().sort_values(ascending=False)

CryoSleep       0.024963
ShoppingMall    0.023927
VIP             0.023352
HomePlanet      0.023122
Name            0.023007
Cabin           0.022892
VRDeck          0.021627
Spa             0.021051
FoodCourt       0.021051
Destination     0.020936
RoomService     0.020821
Age             0.020591
PassengerId     0.000000
Transported     0.000000
dtype: float64

In [8]:
for col in train.columns:
    print("\n", col)
    print(train[col].head())
    print("Unique:", train[col].nunique())


 PassengerId
0    0001_01
1    0002_01
2    0003_01
3    0003_02
4    0004_01
Name: PassengerId, dtype: object
Unique: 8693

 HomePlanet
0    Europa
1     Earth
2    Europa
3    Europa
4     Earth
Name: HomePlanet, dtype: object
Unique: 3

 CryoSleep
0    False
1    False
2    False
3    False
4    False
Name: CryoSleep, dtype: object
Unique: 2

 Cabin
0    B/0/P
1    F/0/S
2    A/0/S
3    A/0/S
4    F/1/S
Name: Cabin, dtype: object
Unique: 6560

 Destination
0    TRAPPIST-1e
1    TRAPPIST-1e
2    TRAPPIST-1e
3    TRAPPIST-1e
4    TRAPPIST-1e
Name: Destination, dtype: object
Unique: 3

 Age
0    39.0
1    24.0
2    58.0
3    33.0
4    16.0
Name: Age, dtype: float64
Unique: 80

 VIP
0    False
1    False
2     True
3    False
4    False
Name: VIP, dtype: object
Unique: 2

 RoomService
0      0.0
1    109.0
2     43.0
3      0.0
4    303.0
Name: RoomService, dtype: float64
Unique: 1273

 FoodCourt
0       0.0
1       9.0
2    3576.0
3    1283.0
4      70.0
Name: FoodCourt, dtype: float6

In [9]:
train["GroupId"] = train["PassengerId"].str.split("_").str[0]
test["GroupId"] = test["PassengerId"].str.split("_").str[0]

train["GroupId"] = train["GroupId"].astype(int)
test["GroupId"] = test["GroupId"].astype(int)

In [10]:
train[["Deck", "CabinNumber", "Side"]] = (
    train["Cabin"].str.split("/", expand=True)
)

test[["Deck", "CabinNumber", "Side"]] = (
    test["Cabin"].str.split("/", expand=True)
)

In [11]:
train["CabinNumber"] = pd.to_numeric(
    train["CabinNumber"],
    errors="coerce"
)

test["CabinNumber"] = pd.to_numeric(
    test["CabinNumber"],
    errors="coerce"
)

In [12]:
print(train.shape)
print(train.isna().mean().sort_values(ascending=False))
print(train[["PassengerId", "GroupId", "Cabin", "Deck", "CabinNumber", "Side"]].head(10))

(8693, 18)
CryoSleep       0.024963
ShoppingMall    0.023927
VIP             0.023352
HomePlanet      0.023122
Name            0.023007
Deck            0.022892
Side            0.022892
CabinNumber     0.022892
Cabin           0.022892
VRDeck          0.021627
FoodCourt       0.021051
Spa             0.021051
Destination     0.020936
RoomService     0.020821
Age             0.020591
PassengerId     0.000000
Transported     0.000000
GroupId         0.000000
dtype: float64
  PassengerId  GroupId  Cabin Deck  CabinNumber Side
0     0001_01        1  B/0/P    B          0.0    P
1     0002_01        2  F/0/S    F          0.0    S
2     0003_01        3  A/0/S    A          0.0    S
3     0003_02        3  A/0/S    A          0.0    S
4     0004_01        4  F/1/S    F          1.0    S
5     0005_01        5  F/0/P    F          0.0    P
6     0006_01        6  F/2/S    F          2.0    S
7     0006_02        6  G/0/S    G          0.0    S
8     0007_01        7  F/3/S    F          3.0

In [13]:
group_sizes = train["GroupId"].value_counts()

train["GroupSize"] = train["GroupId"].map(group_sizes)

test["GroupSize"] = test["GroupId"].map(group_sizes).fillna(1)

In [14]:
spend_cols = [
    "RoomService",
    "FoodCourt",
    "ShoppingMall",
    "Spa",
    "VRDeck"
]

train["TotalSpend"] = train[spend_cols].sum(axis=1)
test["TotalSpend"] = test[spend_cols].sum(axis=1)

In [15]:
train["NoSpend"] = (train["TotalSpend"] == 0).astype(int)
test["NoSpend"] = (test["TotalSpend"] == 0).astype(int)

In [16]:
train["GroupMember"] = (
    train["PassengerId"].str.split("_").str[1].astype(int)
)

test["GroupMember"] = (
    test["PassengerId"].str.split("_").str[1].astype(int)
)

In [17]:
display(
    train[
        [
            "PassengerId",
            "GroupId",
            "GroupSize",
            "GroupMember",
            "Cabin",
            "Deck",
            "CabinNumber",
            "Side",
            "TotalSpend",
            "NoSpend",
            "Transported"
        ]
    ].head(10)
)

,PassengerId,GroupId,GroupSize,GroupMember,Cabin,Deck,CabinNumber,Side,TotalSpend,NoSpend,Transported
0,0001_01,1,1,1,B/0/P,B,0.0,P,0.0,1,False
1,0002_01,2,1,1,F/0/S,F,0.0,S,736.0,0,True
2,0003_01,3,2,1,A/0/S,A,0.0,S,10383.0,0,False
3,0003_02,3,2,2,A/0/S,A,0.0,S,5176.0,0,False
4,0004_01,4,1,1,F/1/S,F,1.0,S,1091.0,0,True
5,0005_01,5,1,1,F/0/P,F,0.0,P,774.0,0,True
6,0006_01,6,2,1,F/2/S,F,2.0,S,1584.0,0,True
7,0006_02,6,2,2,G/0/S,G,0.0,S,0.0,1,True
8,0007_01,7,1,1,F/3/S,F,3.0,S,1018.0,0,True
9,0008_01,8,3,1,B/1/P,B,1.0,P,0.0,1,True


In [18]:
train["LogTotalSpend"] = np.log1p(train["TotalSpend"])
test["LogTotalSpend"] = np.log1p(test["TotalSpend"])

In [19]:
model_features = [
    "HomePlanet",
    "CryoSleep",
    "Destination",
    "VIP",
    "Age",
    "RoomService",
    "FoodCourt",
    "ShoppingMall",
    "Spa",
    "VRDeck",
    "Deck",
    "CabinNumber",
    "Side",
    "GroupSize",
    #"TotalSpend",
    "LogTotalSpend"
    #"NoSpend"
]

X = train[model_features]
y = train["Transported"]

X_test = test[model_features]

In [20]:
categorical_features = [
    "HomePlanet",
    "CryoSleep",
    "Destination",
    "VIP",
    "Deck",
    "Side",
    #"NoSpend"
]

numeric_features = [
    "Age",
    "RoomService",
    "FoodCourt",
    "ShoppingMall",
    "Spa",
    "VRDeck",
    "CabinNumber",
    "GroupSize",
    #"TotalSpend"
    "LogTotalSpend"
]

In [21]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder , StandardScaler

numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])

In [22]:
from sklearn.linear_model import LogisticRegression

log_model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(
        max_iter=1000
    ))
])

In [23]:
from sklearn.model_selection import StratifiedKFold, cross_val_score

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

scores = cross_val_score(
    log_model,
    X,
    y,
    cv=cv,
    scoring="accuracy"
)

print("Accuracy:", scores)
print("Mean Accuracy:", scores.mean())
print("Std:", scores.std())

Accuracy: [0.79183439 0.78378378 0.80046003 0.79631761 0.78653625]
Mean Accuracy: 0.7917864121742387
Std: 0.0061216342027581045


In [24]:
from sklearn.ensemble import RandomForestClassifier

rf_model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(
        n_estimators=300,
        max_depth=None,
        min_samples_leaf=1,
        random_state=42,
        n_jobs=-1
    ))
])

rf_scores = cross_val_score(
    rf_model,
    X,
    y,
    cv=cv,
    scoring="accuracy"
)

print("Accuracy:", rf_scores)
print("Mean Accuracy:", rf_scores.mean())
print("Std:", rf_scores.std())

Accuracy: [0.81253594 0.80103508 0.79988499 0.81242808 0.78596087]
Mean Accuracy: 0.8023689924040045
Std: 0.009815579005967886


In [25]:
pd.crosstab(
    train["CryoSleep"],
    train["NoSpend"],
    normalize="index"
)

NoSpend,0,1
CryoSleep,,
False,0.904762,0.095238
True,0.000000,1.000000


In [26]:
pd.crosstab(
    train["CryoSleep"],
    train["Transported"],
    normalize="index"
)

Transported,False,True
CryoSleep,,
False,0.671079,0.328921
True,0.182417,0.817583


In [27]:
from sklearn.ensemble import GradientBoostingClassifier

gb_model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", GradientBoostingClassifier(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=3,
        random_state=42
    ))
])

gb_scores = cross_val_score(
    gb_model,
    X,
    y,
    cv=cv,
    scoring="accuracy"
)

print("Accuracy:", gb_scores)
print("Mean Accuracy:", gb_scores.mean())
print("Std:", gb_scores.std())

Accuracy: [0.80621047 0.79930995 0.8119609  0.81357883 0.79401611]
Mean Accuracy: 0.8050152495614386
Std: 0.007436893577120992


In [28]:
final_model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", GradientBoostingClassifier(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=3,
        random_state=42
    ))
])

final_model.fit(X, y)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['Age', 'RoomService',
                                                   'FoodCourt', 'ShoppingMall',
                                                   'Spa', 'VRDeck',
                                                   'CabinNumber', 'GroupSize',
                                                   'LogTotalSpend']),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('onehot',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['HomePlanet', 'CryoSleep',
                                                   'Destination', 'VIP', 'Deck',
                                                   'Side'])])),
                ('classifier',
                 GradientBoostingClassifier(learning_rate=0.05,
                                            n_estimators=300,
                                            random_state=42))])

In [29]:
predictions = final_model.predict(X_test)

In [30]:
submission = pd.DataFrame({
    "PassengerId": test["PassengerId"],
    "Transported": predictions
})

submission.to_csv(
    "submission.csv",
    index=False
)

print(submission.head())
print(submission.shape)

  PassengerId  Transported
0     0013_01         True
1     0018_01        False
2     0019_01         True
3     0021_01         True
4     0023_01         True
(4277, 2)
